In [1]:
# Load basic libs
import os
import sys
import pandas as pd
import logging

In [2]:
# 1. Create a master logger
logger = logging.getLogger()
logger.setLevel(logging.DEBUG) # Capture everything at the root level

# 2. Create the File Handler (Logs everything from INFO and above)
file_handler = logging.FileHandler('modres_RNA.log')
file_handler.setLevel(logging.INFO)
file_formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler.setFormatter(file_formatter)

# 3. Create the Console Handler (Logs ONLY WARNING and above to stdout)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.WARNING)
console_formatter = logging.Formatter('%(levelname)s: %(message)s')
console_handler.setFormatter(console_formatter)

# 4. Add both handlers to the logger
logger.addHandler(file_handler)
logger.addHandler(console_handler)

In [4]:
# Gemmi for CIF mol parsing
import gemmi
from collections import Counter
from pathlib import Path
import glob

In [14]:
import gemmi

def get_rna_modifications(cif_file):
    # Read the CIF block
    doc = gemmi.cif.read_file(cif_file)
    block = doc.sole_block()
    
    # Generate Gemmi's high-level Structure object from the block
    st = gemmi.make_structure_from_block(block)
    
    # Standard Canonical Nucleic acids
    canonical_nucleic = {"A", "G", "C", "U", "DA", "DG", "DC", "DT"}
    
    # Standard Amino Acids (20 standard + common variants/hetero-atoms)
    amino_acids = {
        "ARG", "LYS", "ASP", "GLU", "ASN", "GLN", "HIS", "SER", "THR", "TYR", 
        "TRP", "PHE", "CYS", "MET", "LEU", "ILE", "PRO", "VAL", "ALA", "GLY",
        "MSE" # Selenomethionine (common in PDB structures)
    }
    
    # Solvents and crystallization agents to ignore
    common_solvents = {"HOH", "DOD", "WAT", "GOL", "EDO", "SO4", "PO4", "CL", "MG"}

    modifications = set()  # Use a set to prevent duplicate names

    # Loop through models, chains, and residues elegantly
    for model in st:
        for chain in model:
            for res in chain:
                resname = res.name.strip()
                
                # Filter: Must NOT be standard nucleic, NOT an amino acid, and NOT solvent
                if (resname not in canonical_nucleic and 
                    resname not in amino_acids and 
                    resname not in common_solvents):
                    
                    # Ensure it's part of an RNA/DNA polymer or flagged as HETERO
                    modifications.add(resname)
                    
    return list(modifications)

In [15]:
# Update your working directory
work_dir = "/home/labuser/Projects/rcsb_NAdb/structures/"
data_subdir = "CCD_Modified_Nucleotides"


freq = Counter()

for cif_file in glob.glob(
        f"{work_dir}/*.cif"
):

    mods = get_rna_modifications(cif_file)

    freq.update(mods)

In [23]:
freq_df = pd.DataFrame(
    freq.items(),
    columns=[
        "id",
        "frequency"
    ]
)

In [24]:
freq_df

,id,frequency
0,ZN,958
1,MN,644
2,ADP,113
3,CTP,30
4,BU3,3
...,...,...
1120,ET,1
1121,QUW,1
1122,AES,1
1123,SSJ,1


In [20]:
catalog_df = pd.read_csv("/home/labuser/Projects/shrikant_libs/RNAmodViz/modified_nucleotide_catalog.csv")

In [25]:
catalog_df

,id,name,type,formula,formula_weight,one_letter_code,parent_comp_id,synonyms,smiles,inchi,inchikey,atom_count,bond_count,parent_base,build_method,png_file
0,UR3,"""3-METHYLURIDINE-5'-MONOPHOSHATE""","""RNA LINKING""","""C10 H15 N2 O9 P""",338.208,U,U,?,CN1C(=O)C=CN(C1=O)[C@H]2[C@@H]([C@@H]([C@H](O2...,InChI=1S/C10H15N2O9P/c1-11-6(13)2-3-12(10(11)1...,KZSHODFDDQCIMT-ZOQUXTDFSA-N,37,38,Uridine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...
1,OMG,"""O2'-METHYLGUANOSINE-5'-MONOPHOSPHATE""","""RNA LINKING""","""C11 H16 N5 O8 P""",377.247,G,G,?,CO[C@@H]1[C@@H]([C@H](O[C@H]1n2cnc3c2N=C(NC3=O...,InChI=1S/C11H16N5O8P/c1-22-7-6(17)4(2-23-25(19...,YPMKZCOIEXUDSS-KQYNXXCUSA-N,41,43,Guanosine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...
2,GDP,"""GUANOSINE-5'-DIPHOSPHATE""","""RNA LINKING""","""C10 H15 N5 O11 P2""",443.201,G,G,?,c1nc2c(n1[C@H]3[C@@H]([C@@H]([C@H](O3)CO[P@](=...,InChI=1S/C10H15N5O11P2/c11-10-13-7-4(8(18)14-1...,QGWNDRXFNXRZMB-UUOKFMHZSA-N,43,45,Guanosine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...
3,RPC,";cytidine 3',5'-bis(dihydrogen phosphate)\n;","""RNA LINKING""","""C9 H15 N3 O11 P2""",403.176,?,C,?,C1=CN(C(=O)N=C1N)[C@H]2[C@@H]([C@@H]([C@H](O2)...,InChI=1S/C9H15N3O11P2/c10-5-1-2-12(9(14)11-5)8...,MVGIITUCOOTSAP-XVFCMESISA-N,40,41,Cytidine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...
4,A2M,;2'-O-methyladenosine 5'-(dihydrogen phosphate...,"""RNA LINKING""","""C11 H16 N5 O7 P""",361.248,A,A,?,CO[C@@H]1[C@@H]([C@H](O[C@H]1n2cnc3c2ncnc3N)CO...,InChI=1S/C11H16N5O7P/c1-21-8-7(17)5(2-22-24(18...,TVGFEBXIZUYVFR-IOSLPCCCSA-N,40,42,Adenosine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,70U,"""5-(O-METHYLACETO)-2-THIO-2-DEOXY-URIDINE-5'-M...","""RNA LINKING""","""C12 H17 N2 O10 P S""",412.310,U,U,?,COC(=O)CC1=CN(C(=S)NC1=O)[C@H]2[C@@H]([C@@H]([...,InChI=1S/C12H17N2O10PS/c1-22-7(15)2-5-3-14(12(...,RWXSPKNHAJLKOR-PNHWDRBUSA-N,43,44,Uridine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...
67,N79,"""[(1S,5R,6R,8R)-6-(6-aminopurin-9-yl)spiro[2,4...",non-polymer,"""C17 H18 B N5 O8 P""",462.138,A,A,?,[B-]12(c3ccccc3CO1)O[C@@H]4[C@H](O[C@H]([C@@H]...,InChI=1S/C17H18BN5O8P/c19-15-12-16(21-7-20-15)...,WVTCQRMVQMUXKS-LBTDBDNISA-N,50,55,Adenosine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...
68,1MA,"""6-HYDRO-1-METHYLADENOSINE-5'-MONOPHOSPHATE""","""RNA LINKING""","""C11 H16 N5 O7 P""",361.248,A,A,?,[H]/N=C\1/c2c(n(cn2)[C@H]3[C@@H]([C@@H]([C@H](...,InChI=1S/C11H16N5O7P/c1-15-3-14-10-6(9(15)12)1...,BKBYKEWNXKDACS-JOLDIKRXSA-N,40,42,Adenosine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...
69,FMU,"""5-FLUORO-5-METHYLURIDINE-5'-MONOPHOSPHATE""","""RNA LINKING""","""C10 H16 F N2 O9 P""",358.214,N,?,?,C[C@@]1(CN(C(=O)NC1=O)[C@H]2[C@@H]([C@@H]([C@H...,InChI=1S/C10H16FN2O9P/c1-10(11)3-13(9(17)12-8(...,MVSQIWGOIJYVSH-NVABTJCQSA-N,39,40,Unknown,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...


In [32]:
final_df = catalog_df.merge(
    freq_df,
    on="id",
    how="left"
)

In [33]:
final_df

,id,name,type,formula,formula_weight,one_letter_code,parent_comp_id,synonyms,smiles,inchi,inchikey,atom_count,bond_count,parent_base,build_method,png_file,frequency
0,UR3,"""3-METHYLURIDINE-5'-MONOPHOSHATE""","""RNA LINKING""","""C10 H15 N2 O9 P""",338.208,U,U,?,CN1C(=O)C=CN(C1=O)[C@H]2[C@@H]([C@@H]([C@H](O2...,InChI=1S/C10H15N2O9P/c1-11-6(13)2-3-12(10(11)1...,KZSHODFDDQCIMT-ZOQUXTDFSA-N,37,38,Uridine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...,1.0
1,OMG,"""O2'-METHYLGUANOSINE-5'-MONOPHOSPHATE""","""RNA LINKING""","""C11 H16 N5 O8 P""",377.247,G,G,?,CO[C@@H]1[C@@H]([C@H](O[C@H]1n2cnc3c2N=C(NC3=O...,InChI=1S/C11H16N5O8P/c1-22-7-6(17)4(2-23-25(19...,YPMKZCOIEXUDSS-KQYNXXCUSA-N,41,43,Guanosine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...,17.0
2,GDP,"""GUANOSINE-5'-DIPHOSPHATE""","""RNA LINKING""","""C10 H15 N5 O11 P2""",443.201,G,G,?,c1nc2c(n1[C@H]3[C@@H]([C@@H]([C@H](O3)CO[P@](=...,InChI=1S/C10H15N5O11P2/c11-10-13-7-4(8(18)14-1...,QGWNDRXFNXRZMB-UUOKFMHZSA-N,43,45,Guanosine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...,13.0
3,RPC,";cytidine 3',5'-bis(dihydrogen phosphate)\n;","""RNA LINKING""","""C9 H15 N3 O11 P2""",403.176,?,C,?,C1=CN(C(=O)N=C1N)[C@H]2[C@@H]([C@@H]([C@H](O2)...,InChI=1S/C9H15N3O11P2/c10-5-1-2-12(9(14)11-5)8...,MVGIITUCOOTSAP-XVFCMESISA-N,40,41,Cytidine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...,1.0
4,A2M,;2'-O-methyladenosine 5'-(dihydrogen phosphate...,"""RNA LINKING""","""C11 H16 N5 O7 P""",361.248,A,A,?,CO[C@@H]1[C@@H]([C@H](O[C@H]1n2cnc3c2ncnc3N)CO...,InChI=1S/C11H16N5O7P/c1-21-8-7(17)5(2-22-24(18...,TVGFEBXIZUYVFR-IOSLPCCCSA-N,40,42,Adenosine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...,22.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,70U,"""5-(O-METHYLACETO)-2-THIO-2-DEOXY-URIDINE-5'-M...","""RNA LINKING""","""C12 H17 N2 O10 P S""",412.310,U,U,?,COC(=O)CC1=CN(C(=S)NC1=O)[C@H]2[C@@H]([C@@H]([...,InChI=1S/C12H17N2O10PS/c1-22-7(15)2-5-3-14(12(...,RWXSPKNHAJLKOR-PNHWDRBUSA-N,43,44,Uridine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...,NaN
67,N79,"""[(1S,5R,6R,8R)-6-(6-aminopurin-9-yl)spiro[2,4...",non-polymer,"""C17 H18 B N5 O8 P""",462.138,A,A,?,[B-]12(c3ccccc3CO1)O[C@@H]4[C@H](O[C@H]([C@@H]...,InChI=1S/C17H18BN5O8P/c19-15-12-16(21-7-20-15)...,WVTCQRMVQMUXKS-LBTDBDNISA-N,50,55,Adenosine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...,2.0
68,1MA,"""6-HYDRO-1-METHYLADENOSINE-5'-MONOPHOSPHATE""","""RNA LINKING""","""C11 H16 N5 O7 P""",361.248,A,A,?,[H]/N=C\1/c2c(n(cn2)[C@H]3[C@@H]([C@@H]([C@H](...,InChI=1S/C11H16N5O7P/c1-15-3-14-10-6(9(15)12)1...,BKBYKEWNXKDACS-JOLDIKRXSA-N,40,42,Adenosine,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...,9.0
69,FMU,"""5-FLUORO-5-METHYLURIDINE-5'-MONOPHOSPHATE""","""RNA LINKING""","""C10 H16 F N2 O9 P""",358.214,N,?,?,C[C@@]1(CN(C(=O)NC1=O)[C@H]2[C@@H]([C@@H]([C@H...,InChI=1S/C10H16FN2O9P/c1-10(11)3-13(9(17)12-8(...,MVSQIWGOIJYVSH-NVABTJCQSA-N,39,40,Unknown,SMILES,/home/labuser/Projects/shrikant_libs/modres_he...,1.0
